In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from pathlib import Path
from collections import defaultdict, Counter
import json

# Verify GPU
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("TF version:", tf.__version__)

In [ ]:
# Paths and parameters
BASE_DIR = Path('/content/drive/MyDrive/Splits2')
TRAIN_DIR = BASE_DIR / 'train'
VAL_DIR = BASE_DIR / 'val'
TEST_DIR = BASE_DIR / 'test'

CHECKPOINT_DIR = Path('/content/drive/MyDrive/augmentation_data/checkpoints')
HISTORY_PATH = Path('/content/drive/MyDrive/augmentation_data/history_v3.json')
MODEL_PATH = Path('/content/drive/MyDrive/augmentation_data/tank_classifier_v3.keras')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (682, 1024)
BATCH_SIZE = 4
SEED = 42
NUM_CLASSES = 3
AUTOTUNE = tf.data.AUTOTUNE

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR, labels='inferred', label_mode='int',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
    crop_to_aspect_ratio=True,
)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, labels='inferred', label_mode='int',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
    crop_to_aspect_ratio=True,
)
class_names = val_ds.class_names
print(f'Classes: {class_names}')

val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# Augmentation - stronger for T-72 to reduce T-72/T-80 confusion
default_aug = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])
t72_aug = keras.Sequential([
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
])

def load_class_ds(class_path, label, augmentation):
    """Load one class folder, apply augmentation, tag with label."""
    ds = keras.utils.image_dataset_from_directory(
        class_path, labels=None, image_size=IMG_SIZE,
        batch_size=None, shuffle=True, seed=SEED,
    )
    return ds.map(lambda img: (augmentation(img, training=True),
                               tf.cast(label, tf.int32)))

t72_ds = load_class_ds(TRAIN_DIR / 't72', 0, t72_aug)
t80_ds = load_class_ds(TRAIN_DIR / 't80', 1, default_aug)
t90_ds = load_class_ds(TRAIN_DIR / 't90', 2, default_aug)

train_ds = (t72_ds
            .concatenate(t80_ds)
            .concatenate(t90_ds)
            .shuffle(buffer_size=1000, seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

# Build the model: EfficientNetB4 with top 40 backbone layers unfrozen
base_model = keras.applications.EfficientNetB4(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

trainable_count = sum(layer.trainable for layer in base_model.layers)
print(f'Trainable layers in base_model: {trainable_count}/{len(base_model.layers)}')

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)   # keep BN layers in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu',
                 kernel_regularizer=keras.regularizers.l2(1e-2))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs, name='tank_classifier_v3')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

# T-80 weighted higher to penalise T-72/T-80 confusion
class_weight = {0: 1.0, 1: 1.5, 2: 1.0}

class SaveHistoryCallback(keras.callbacks.Callback):
    """Persist training history to disk after every epoch."""
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath

    def on_epoch_end(self, epoch, logs=None):
        with open(self.filepath, 'w') as f:
            json.dump(self.model.history.history, f)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15, restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'v3_epoch_{epoch:02d}_val{val_accuracy:.3f}.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1,
    ),
    SaveHistoryCallback(filepath=str(HISTORY_PATH)),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weight,
    callbacks=callbacks,
)

test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy: {test_acc:.4f}')

model.save(MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training and Validation Accuracy')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training and Validation Loss')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

# Frame-level evaluation
y_true, y_pred = [], []
for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
    cmap='Blues', values_format='d',
)
plt.title('Confusion Matrix (frame-level)')
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

# Sequence-level majority voting across 40-frame sequences
SEQUENCE_SIZE = 40

def evaluate_with_majority_vote(model, test_dir, class_names, img_size):
    y_true_seq, y_pred_seq = [], []
    for class_idx, class_name in enumerate(class_names):
        files = sorted(f for f in (Path(test_dir) / class_name).iterdir()
                       if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp'))
        sequences = [files[i:i + SEQUENCE_SIZE]
                     for i in range(0, len(files), SEQUENCE_SIZE)]
        for seq in sequences:
            frame_preds = []
            for img_path in seq:
                img = keras.utils.load_img(img_path, target_size=img_size)
                arr = keras.utils.img_to_array(img)
                arr = keras.applications.efficientnet.preprocess_input(arr)
                arr = np.expand_dims(arr, axis=0)
                frame_preds.append(np.argmax(model.predict(arr, verbose=0)))
            majority = Counter(frame_preds).most_common(1)[0][0]
            y_true_seq.append(class_idx)
            y_pred_seq.append(majority)
    return y_true_seq, y_pred_seq

y_true_seq, y_pred_seq = evaluate_with_majority_vote(model, TEST_DIR, class_names, IMG_SIZE)
seq_acc = np.mean(np.array(y_true_seq) == np.array(y_pred_seq))
print(f'Sequence-level accuracy (majority vote): {seq_acc:.4f}')

cm_seq = confusion_matrix(y_true_seq, y_pred_seq)
ConfusionMatrixDisplay(confusion_matrix=cm_seq, display_labels=class_names).plot(
    cmap='Blues', values_format='d',
)
plt.title('Confusion Matrix (Majority Vote)')
plt.show()
print('\nClassification Report (Majority Vote):')
print(classification_report(y_true_seq, y_pred_seq, target_names=class_names))

# Reconstruct file order Keras uses internally for test_ds
test_file_paths = []
for class_name in sorted(class_names):
    files = sorted(f for f in (TEST_DIR / class_name).iterdir()
                   if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp'))
    test_file_paths.extend(str(f) for f in files)
print(f'Test files found: {len(test_file_paths)}, test images in test_ds: {len(y_true)}')

# Collect misclassified images
misclassified = []
file_index = 0
for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    pred_labels = np.argmax(predictions, axis=1)
    confidences = np.max(predictions, axis=1)
    for i in range(images.shape[0]):
        true_label = int(labels[i].numpy())
        pred_label = int(pred_labels[i])
        if true_label != pred_label:
            misclassified.append({
                'file_path': test_file_paths[file_index],
                'true_label': class_names[true_label],
                'pred_label': class_names[pred_label],
                'confidence': float(confidences[i]),
            })
        file_index += 1

print(f'Misclassified images: {len(misclassified)}')
for item in misclassified[:20]:
    print(f"  {Path(item['file_path']).name}: "
          f"true={item['true_label']}, pred={item['pred_label']}, "
          f"conf={item['confidence']:.3f}")


def show_misclassified_images(misclassified, num_images=15):
    """Show a balanced sample of misclassified images across true classes."""
    by_class = defaultdict(list)
    for item in misclassified:
        by_class[item['true_label']].append(item)

    sampled = []
    per_class = max(1, num_images // len(by_class))
    for items in by_class.values():
        sampled.extend(items[:per_class])
    sampled = sampled[:num_images]

    plt.figure(figsize=(15, 10))
    for i, item in enumerate(sampled):
        img = keras.utils.load_img(item['file_path'], target_size=IMG_SIZE)
        plt.subplot(3, 5, i + 1)
        plt.imshow(keras.utils.img_to_array(img).astype('uint8'))
        plt.title(f"True: {item['true_label']}\n"
                  f"Pred: {item['pred_label']}\n"
                  f"Conf: {item['confidence']:.2f}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_misclassified_images(misclassified, num_images=15)

# Grad-CAM: find last conv layer in backbone, split model into feature extractor
# and classifier head, generate heatmaps for misclassified images
last_conv_layer_name = next(
    layer.name for layer in reversed(base_model.layers)
    if isinstance(layer, layers.Conv2D)
)
print(f'Last conv layer: {last_conv_layer_name}')
last_conv_layer = base_model.get_layer(last_conv_layer_name)

feature_extractor = keras.Model(
    inputs=base_model.input,
    outputs=[last_conv_layer.output, base_model.output],
)

classifier_input = keras.Input(shape=base_model.output.shape[1:])
x = classifier_input
passed_base = False
for layer in model.layers:
    if passed_base:
        x = layer(x)
    if layer is base_model:
        passed_base = True
classifier_model = keras.Model(classifier_input, x)


def make_gradcam_heatmap(img_array_preprocessed, pred_index=None):
    with tf.GradientTape() as tape:
        last_conv_output, base_output = feature_extractor(
            img_array_preprocessed, training=False,
        )
        tape.watch(last_conv_output)
        preds = classifier_model(base_output, training=False)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_output = last_conv_output[0]
    heatmap = tf.squeeze(last_conv_output @ pooled_grads[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap /= max_val
    return heatmap.numpy()


def get_gradcam_overlay(img_path, alpha=0.4):
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras.utils.img_to_array(img).astype('float32')
    original_img = img_array.astype('uint8')

    input_array = np.expand_dims(img_array.copy(), axis=0)
    input_array = keras.applications.efficientnet.preprocess_input(input_array)

    preds = model.predict(input_array, verbose=0)
    pred_index = int(np.argmax(preds[0]))

    heatmap = make_gradcam_heatmap(input_array, pred_index=pred_index)
    heatmap = np.uint8(255 * heatmap)
    jet_colors = mpl_cm.get_cmap('jet')(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((original_img.shape[1], original_img.shape[0]))
    jet_heatmap = keras.utils.img_to_array(jet_heatmap)

    superimposed = jet_heatmap * alpha + original_img
    return np.clip(superimposed / 255.0, 0, 1)


def show_gradcam_images(misclassified, num_images=5):
    num_images = min(num_images, len(misclassified))
    cols = 5
    rows = int(np.ceil(num_images / cols))
    plt.figure(figsize=(5 * cols, 4 *